<!-- Chart 01 removed — raw event counts, no actionable business question -->

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs('../data/processed', exist_ok=True)
os.makedirs('charts', exist_ok=True)

## 1.1 Load + Inspect

In [2]:
df = pd.read_csv('../events.csv')

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.info()

Shape: (885129, 9)

Columns: ['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 885129 entries, 0 to 885128
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   event_time     885129 non-null  object 
 1   event_type     885129 non-null  object 
 2   product_id     885129 non-null  int64  
 3   category_id    885129 non-null  int64  
 4   category_code  648910 non-null  object 
 5   brand          672765 non-null  object 
 6   price          885129 non-null  float64
 7   user_id        885129 non-null  int64  
 8   user_session   884964 non-null  object 
dtypes: float64(1), int64(3), object(5)
memory usage: 60.8+ MB


In [3]:
df.describe(include='all')

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
count,885129,885129,8.851290e+05,8.851290e+05,648910,672765,885129.000000,8.851290e+05,884964
unique,845041,3,NaN,NaN,107,999,NaN,NaN,490398
top,2021-02-04 21:48:32 UTC,view,NaN,NaN,computers.components.videocards,asus,NaN,NaN,nFlhu5QzOd
freq,18,793748,NaN,NaN,116717,27706,NaN,NaN,572
mean,NaN,NaN,1.906621e+06,2.144423e+18,NaN,NaN,146.328713,1.515916e+18,NaN
std,NaN,NaN,1.458708e+06,6.165105e+14,NaN,NaN,296.807683,3.554165e+07,NaN
min,NaN,NaN,1.020000e+02,2.144416e+18,NaN,NaN,0.220000,1.515916e+18,NaN
25%,NaN,NaN,6.988030e+05,2.144416e+18,NaN,NaN,26.460000,1.515916e+18,NaN
50%,NaN,NaN,1.452883e+06,2.144416e+18,NaN,NaN,65.710000,1.515916e+18,NaN
75%,NaN,NaN,3.721194e+06,2.144416e+18,NaN,NaN,190.490000,1.515916e+18,NaN


In [4]:
print("Null counts:")
print(df.isnull().sum())
print(f"\nEvent types: {df['event_type'].value_counts().to_dict()}")

Null counts:
event_time            0
event_type            0
product_id            0
category_id           0
category_code    236219
brand            212364
price                 0
user_id               0
user_session        165
dtype: int64

Event types: {'view': 793748, 'cart': 54035, 'purchase': 37346}


## 1.2 Parse Timestamps

In [5]:
df['event_time'] = pd.to_datetime(df['event_time'], utc=True)
df['event_date'] = df['event_time'].dt.date
df['event_month'] = df['event_time'].dt.to_period('M').dt.to_timestamp()

print(f"Date range: {df['event_date'].min()} → {df['event_date'].max()}")
print(f"Months covered: {df['event_month'].nunique()}")

Date range: 2020-09-24 → 2021-02-28
Months covered: 6


C:\Users\Inspiron16\AppData\Local\Temp\claude\ipykernel_23108\2747871356.py:3: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['event_month'] = df['event_time'].dt.to_period('M').dt.to_timestamp()


## 1.3 Parse Categories

In [6]:
cat_split = df['category_code'].str.split('.', expand=True)
df['category_l1'] = cat_split[0].fillna('unknown')
df['category_l2'] = cat_split[1].fillna('unknown')

print("Top category_l2 values:")
print(df['category_l2'].value_counts().head(10))

Top category_l2 values:
category_l2
unknown        236219
components     202246
telephone       84360
peripherals     78357
cartrige        38725
audio           37270
accessories     35433
tools           31780
notebook        25032
video           24047
Name: count, dtype: int64


## 1.4 Handle Brand Nulls

In [7]:
brand_null_count = df['brand'].isnull().sum()
brand_null_pct = 100 * brand_null_count / len(df)
print(f"Brand nulls: {brand_null_count:,} ({brand_null_pct:.1f}%)")
print("Keeping nulls — they are informative (often occur in cart/purchase events)")

# Flag as 'unknown' but retain the original null pattern for analysis
df['brand'] = df['brand'].fillna('unknown')

Brand nulls: 212,364 (24.0%)
Keeping nulls — they are informative (often occur in cart/purchase events)


## 1.5 Price Outliers

In [8]:
print("Price distribution:")
print(df['price'].describe())

# Flag zero/negative prices
bad_price_mask = df['price'] <= 0
print(f"\nRows with price <= 0: {bad_price_mask.sum()}")
df['price_flag'] = bad_price_mask

# Cap at 99th percentile
p99 = df['price'].quantile(0.99)
print(f"99th percentile price: ${p99:,.2f}")
df['price'] = df['price'].clip(upper=p99)
print(f"Price range after cap: ${df['price'].min():.2f} – ${df['price'].max():.2f}")

Price distribution:
count    885129.000000
mean        146.328713
std         296.807683
min           0.220000
25%          26.460000
50%          65.710000
75%         190.490000
max       64771.060000
Name: price, dtype: float64

Rows with price <= 0: 0
99th percentile price: $889.98
Price range after cap: $0.22 – $889.98


## 1.6 Duplicate Check

In [9]:
key_cols = ['user_id', 'user_session', 'event_type', 'product_id', 'event_time']
dupes = df.duplicated(subset=key_cols).sum()
print(f"Duplicate rows on key columns: {dupes}")

if dupes > 0:
    df = df.drop_duplicates(subset=key_cols)
    print(f"Rows after dedup: {len(df):,}")
else:
    print("No duplicates — events table is atomic.")

Duplicate rows on key columns: 655


Rows after dedup: 884,474


## 1.7 `remove_from_cart` Events

In [10]:
rfc_count = (df['event_type'] == 'remove_from_cart').sum()
print(f"remove_from_cart events: {rfc_count:,} ({100*rfc_count/len(df):.1f}%)")
print("Decision: KEEP in dataset. Excluded from funnel queries (documented in SQL).")
print("These events show browse/consideration behavior — useful for future cart-abandonment analysis.")

remove_from_cart events: 0 (0.0%)
Decision: KEEP in dataset. Excluded from funnel queries (documented in SQL).
These events show browse/consideration behavior — useful for future cart-abandonment analysis.


## 1.8 Save Parquet

In [11]:
out_path = '../data/processed/events_clean.parquet'
df.to_parquet(out_path, index=False)
print(f"Saved: {out_path}")
print(f"Final shape: {df.shape}")

Saved: ../data/processed/events_clean.parquet
Final shape: (884474, 14)


<!-- Chart 01 removed — raw event counts, no actionable business question -->

In [12]:
# Chart 01 removed — see implementation notes
pass

## Summary

| Step | Result |
|------|--------|
| Raw rows loaded | See `.shape` above |
| Timestamp parsing | UTC-aware, `event_date` + `event_month` derived |
| Category parsing | `category_l1`, `category_l2` extracted |
| Brand nulls | Filled as `'unknown'` — not dropped |
| Price cap | 99th percentile applied |
| Duplicates | Checked on 5-column key |
| `remove_from_cart` | Retained, excluded only from funnel SQL |
| Output | `../data/processed/events_clean.parquet` |